# Validate and publish one lightweight result bundle

Dry-run is the default. No Git branch, commit, push, or pull request is created until you explicitly set `PUBLISH_RESULTS=True` and `DRY_RUN=False`.


In [ ]:
import importlib.util
import os
import subprocess
import sys
from pathlib import Path

SMOKE_TEST = os.environ.get("SMOKE_TEST", "0").lower() in {"1", "true", "yes", "on"}
try:
    IS_COLAB = importlib.util.find_spec("google.colab") is not None
except ModuleNotFoundError:
    IS_COLAB = False
REPOSITORY_URL = os.environ.get(
    "BENCHMARK_REPOSITORY_URL",
    "https://github.com/Harryphan72007/aerial-object-detection-benchmark.git",
)
REPOSITORY_BRANCH = os.environ.get("BENCHMARK_REPOSITORY_BRANCH", "main")
if IS_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    REPO_DIR = Path("/content/aerial-object-detection-benchmark")
    if not (REPO_DIR / ".git").is_dir():
        subprocess.run(
            ["git", "clone", "--branch", REPOSITORY_BRANCH, REPOSITORY_URL, str(REPO_DIR)],
            check=True,
        )
else:
    REPO_DIR = Path(os.environ.get("BENCHMARK_REPO_ROOT", Path.cwd())).resolve()
if not (REPO_DIR / "pyproject.toml").is_file():
    raise RuntimeError(f"Repository root is invalid: {REPO_DIR}")
os.chdir(REPO_DIR)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))
if IS_COLAB:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-r", "requirements-dataset-colab.txt"],
        check=True,
    )
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "."], check=True)
DRIVE_ROOT = os.environ.get(
    "VISDRONE_DRIVE_ROOT",
    "/content/drive/MyDrive/visdrone_architecture_benchmark"
    if IS_COLAB
    else str(REPO_DIR / ".notebook-smoke"),
)
from src.paths import ProjectPaths
from src.reproducibility import seed_everything
from src.utils.environment import collect_environment
paths = ProjectPaths.from_value(DRIVE_ROOT).create()
seed_everything(42)
print({"repo": str(REPO_DIR), "storage": str(paths.root), "smoke_test": SMOKE_TEST})
collect_environment()


## Configuration

Edit this cell only.


In [ ]:
MODEL_ID = "rtdetrv2_l"
DATASET_TRACK = "2class"
RUN_ID = ""                 # Usually leave blank; required only if discovery is ambiguous.
RESULT_BUNDLE_ID = ""       # Usually leave blank; an existing matching bundle is reused.
CREATE_BUNDLE = True
PUBLISH_RESULTS = False
DRY_RUN = True
GIT_USER_NAME = ""
GIT_USER_EMAIL = ""


## Discover measured artifacts and choose one final run


In [ ]:
import json
import pandas as pd
from datetime import datetime, timezone
from src.benchmark_status import discover_model_status
from src.result_export import validate_bundle
from src.training.checkpointing import RunRegistry
from src.utils.serialization import read_yaml

status = discover_model_status(DRIVE_ROOT, MODEL_ID, REPO_DIR)
print(json.dumps(status, indent=2))
if not SMOKE_TEST and (
    status["evaluation_status"] != "COMPLETE"
    or status["report_status"] != "COMPLETE"
):
    raise RuntimeError("Run notebooks 07 and 10 before creating a publication bundle.")
registry = RunRegistry(paths)
candidates = []
for run in registry.list_available_runs(MODEL_ID, DATASET_TRACK, status="completed"):
    run_dir = Path(run.get("run_dir") or paths.final_checkpoints / MODEL_ID / run["run_id"])
    config = run_dir / "training_config.yaml"
    metrics = list(paths.evaluation.glob(f"{run['run_id']}__res*__metrics.json"))
    if config.exists() and read_yaml(config).get("run_kind") == "final_complete_official_train" and metrics:
        candidates.append({**run, "run_dir": str(run_dir), "evaluation_files": len(metrics)})
display(pd.DataFrame([
    {key: row.get(key) for key in ("run_id", "created_at", "best_validation_map", "evaluation_files")}
    for row in candidates
]))
if RUN_ID:
    selected = [row for row in candidates if row["run_id"] == RUN_ID]
elif len(candidates) == 1:
    selected = candidates
elif len(candidates) > 1:
    raise RuntimeError("Multiple compatible runs found. Copy one run_id into RUN_ID and rerun.")
elif SMOKE_TEST:
    selected = []
else:
    raise RuntimeError("No completed evaluated final run was found.")
SELECTED_RUN_ID = selected[0]["run_id"] if selected else None
matching_bundles = []
for manifest_path in paths.result_bundles.glob("*/bundle_manifest.json"):
    manifest = json.loads(manifest_path.read_text())
    if (
        manifest.get("model_id") == MODEL_ID
        and manifest.get("dataset_track") == DATASET_TRACK
        and manifest.get("run_id") == SELECTED_RUN_ID
        and not validate_bundle(manifest_path.parent)
    ):
        matching_bundles.append(manifest_path.parent)
existing_bundle = max(matching_bundles, key=lambda item: item.stat().st_mtime) if matching_bundles else None
RESULT_BUNDLE_ID = RESULT_BUNDLE_ID or (
    existing_bundle.name if existing_bundle
    else f"{MODEL_ID}__{DATASET_TRACK}__{datetime.now(timezone.utc):%Y%m%d_%H%M%S}"
)
print("Selected run:", SELECTED_RUN_ID or "SMOKE_TEST: none required")
print("Recommended bundle ID:", RESULT_BUNDLE_ID)


## Create or reuse the lightweight bundle


In [ ]:
from src.result_export import create_result_bundle
bundle_path = paths.result_bundles / RESULT_BUNDLE_ID
if SMOKE_TEST:
    print("SMOKE_TEST: bundle creation skipped.")
elif CREATE_BUNDLE and not bundle_path.exists():
    bundle_path = create_result_bundle(
        DRIVE_ROOT,
        DATASET_TRACK,
        REPO_DIR,
        RESULT_BUNDLE_ID,
        model_id=MODEL_ID,
        run_id=SELECTED_RUN_ID,
    )
elif not bundle_path.exists():
    raise FileNotFoundError(bundle_path)
print("Bundle:", bundle_path)


## Validate and preview—no Git mutation


In [ ]:
from src.result_export import export_bundle, validate_bundle
if SMOKE_TEST:
    print("SMOKE_TEST: validation and dry-run publishing imports passed.")
else:
    errors = validate_bundle(bundle_path)
    if errors:
        raise RuntimeError("Bundle validation failed:\n- " + "\n- ".join(errors))
    preview = export_bundle(
        DRIVE_ROOT,
        RESULT_BUNDLE_ID,
        REPO_DIR,
        dry_run=True,
    )
    print("Files that will be copied:")
    for item in preview["preview"]:
        if item["action"] == "copy":
            print(" +", item["destination"])
    print("Files that will be excluded:")
    for item in preview["preview"]:
        if item["action"] == "exclude":
            print(" -", item["source"])
    print("Total file count:", preview["file_count"])
    print("Total bundle size:", preview["total_size_bytes"], "bytes")
    print("Target Git branch:", preview["target_branch"])
    print("Target repository paths:", preview["target_bundle"], preview["latest_manifest"])
    print("Validation result:", preview["validation"])
    print("Git diff preview:", *preview["projected_git_diff"], sep="\n  ")
    if DRY_RUN:
        print("DRY RUN COMPLETE: repository files, Git index, remote, and PR were not changed.")


## Explicit publishing step


In [ ]:
if PUBLISH_RESULTS:
    if DRY_RUN:
        raise RuntimeError("Set DRY_RUN=False only after reviewing the preview.")
    if not GIT_USER_NAME or not GIT_USER_EMAIL:
        raise RuntimeError("Set GIT_USER_NAME and GIT_USER_EMAIL in the configuration cell.")
    if subprocess.run(["git", "status", "--porcelain"], capture_output=True, text=True, check=True).stdout.strip():
        raise RuntimeError("Repository is not clean. Preserve or remove unrelated changes before publishing.")
    subprocess.run(["gh", "auth", "status"], check=True)  # Checks auth; never prints a token.
    subprocess.run(["git", "fetch", "origin"], check=True)
    branch = "experiment-results"
    remote_exists = subprocess.run(
        ["git", "ls-remote", "--exit-code", "--heads", "origin", branch],
        capture_output=True,
    ).returncode == 0
    base_ref = f"origin/{branch}" if remote_exists else "origin/main"
    subprocess.run(["git", "checkout", "-B", branch, base_ref], check=True)
    subprocess.run(["git", "config", "user.name", GIT_USER_NAME], check=True)
    subprocess.run(["git", "config", "user.email", GIT_USER_EMAIL], check=True)
    export_bundle(DRIVE_ROOT, RESULT_BUNDLE_ID, REPO_DIR, dry_run=False)
    subprocess.run(
        [sys.executable, "-m", "scripts.validate_results", "--repo-results", "results"],
        check=True,
    )
    subprocess.run(["git", "add", "--", "results"], check=True)
    subprocess.run(["git", "diff", "--cached", "--stat"], check=True)
    subprocess.run(["git", "diff", "--cached", "--name-status"], check=True)
    staged = subprocess.run(
        ["git", "diff", "--cached", "--name-only"],
        capture_output=True, text=True, check=True,
    ).stdout.splitlines()
    if not staged or any(not path.startswith("results/") for path in staged):
        raise RuntimeError("Staging safety check failed: only results/ may be committed.")
    commit_message = f"results({MODEL_ID}): add 2-class LR benchmark results"
    subprocess.run(["git", "commit", "-m", commit_message], check=True)
    subprocess.run(["git", "push", "-u", "origin", branch], check=True)
    title = f"Results: {MODEL_ID} VisDrone 2-class LR benchmark"
    bundle_manifest = json.loads((bundle_path / "bundle_manifest.json").read_text())
    metric_payload = json.loads((bundle_path / "metrics" / "final_metrics.json").read_text())
    metric = metric_payload["evaluations"][0]
    dataset_provenance = json.loads(
        (bundle_path / "provenance" / "dataset_hashes.json").read_text()
    )
    environment = json.loads(
        (bundle_path / "provenance" / "environment_summary.json").read_text()
    )
    statistics = dataset_provenance.get("split_statistics", {})
    train_images = statistics.get("official_full_train.json", {}).get("images", "recorded in bundle")
    validation_images = statistics.get("official_validation.json", {}).get("images", "recorded in bundle")
    body = (
        f"Model: `{MODEL_ID}`\n\n"
        f"Run: `{SELECTED_RUN_ID}`\n\n"
        f"Selected LR: `{bundle_manifest['selected_learning_rate']}`\n\n"
        "Search: LR-only logarithmic-grid successive halving at epochs 2/5/10/15.\n\n"
        "Final epoch budget: `25`\n\n"
        f"Final training images: `{train_images}`\n\n"
        f"Official validation images: `{validation_images}`\n\n"
        f"mAP50-95: `{metric.get('mAP')}`\n\n"
        f"APtiny: `{metric.get('APtiny')}`\n\n"
        f"Training time (seconds): `{metric.get('total_training_seconds')}`\n\n"
        f"GPU: `{metric.get('evaluation_hardware') or environment.get('gpu_name')}`\n\n"
        "Known limitations: single seed (42), one model-day controlled benchmark.\n\n"
        f"Bundle: `results/bundles/{RESULT_BUNDLE_ID}`\n\n"
        "Checkpoints, datasets, raw predictions, and credentials are excluded."
    )
    existing_pr = subprocess.run(
        ["gh", "pr", "list", "--head", branch, "--base", "main",
         "--state", "open", "--json", "url", "--jq", ".[0].url"],
        capture_output=True, text=True, check=True,
    ).stdout.strip()
    if existing_pr:
        print("Existing pull request:", existing_pr)
    else:
        subprocess.run(
            ["gh", "pr", "create", "--base", "main", "--head", branch,
             "--title", title, "--body", body],
            check=True,
        )
else:
    print("Publishing is OFF. Set PUBLISH_RESULTS=True and DRY_RUN=False only after review.")
